# N3DV Multi-View Frame Decoding Time Benchmark (PyAV)

Measures **synchronized multi-view video decoding time** for the N3DV
(Neural 3D Video / DyNeRF) dataset, swept over:

1. **Frame quality** — re-encode each camera stream at several H.264 CRF /
   resolution settings, then time decoding it back.
2. **Camera count** — `1, 2, 4, 8, 16, 20` cameras.
3. **Parallel mode** — `sync` (baseline), `thread`, `process` workers.
4. **Worker count** — fixed sweep + `"auto"` (= one worker per camera, the
   classic DataLoader `num_workers = #views`).

**Synchronized multi-view scenario.** A streaming 3DGS pipeline (QUEEN-style)
consumes one *timestep* at a time: at timestep `t` it needs frame `t` from
**every** training camera before it can fit/update Gaussians. Workers decode
their assigned views; a **barrier** blocks each timestep's bundle until all
workers have produced frame `t`, then `t+1` starts — exactly how a DataLoader
collates one batch from `num_workers` before the next.

**Timing is split into three phases:**

| Phase | What it covers |
|-------|----------------|
| `T_open`   | Open all streams + init decoders / spin up workers (one-time) |
| `T_first`  | Decode the **first** synchronized multi-view bundle (cold path) |
| `T_steady` | Per-bundle latency **after warm-up** (median over steady timesteps) |

**Decoder: PyAV only** (`av`, libav bindings). OpenCV and decord are
intentionally excluded — we want the direct ffmpeg/libav decode path with
explicit per-frame control so the three phases isolate cleanly. The parallel
backends live in [`mv_parallel_decode.py`](./mv_parallel_decode.py).

Everything is path-parameterized in the **Config** cell — point `N3DV_SCENE_DIR`
at a real scene or leave it empty to run on synthetic videos.

> **Environment.** Run with the `3dgs` conda env
> (`/home/hheo/miniforge3/envs/3dgs/bin/python`). Deps installed there:
> `ffmpeg`, `av`, `pandas`, `numpy`, `matplotlib`.

## Config

In [ ]:
from pathlib import Path

# --- Data location (parameterized; fill in when real data is ready) ---------
# N3DV scene dir is expected to contain per-camera videos: cam00.mp4 .. camNN.mp4
# (the original DyNeRF release) OR cam00/cam01/... subdirs of extracted PNGs.
N3DV_SCENE_DIR = "/home/hheo/data/dynerf/n3dv/cut_roasted_beef/videos"  # empty -> synthetic

# Where to write re-encoded quality variants + synthetic data + result csv.
WORK_DIR = Path("./n3dv_decode_bench")

# --- Sweep axes ------------------------------------------------------------
# Quality variants: H.264 CRF (lower = better quality / bigger file) and an
# optional downscale factor. label -> dict(crf=..., scale=...)
QUALITY_VARIANTS = {
    "lossless":  dict(crf=0,  scale=1.0),
    "high_q18":  dict(crf=18, scale=1.0),
    "med_q28":   dict(crf=28, scale=1.0),
    "low_q38":   dict(crf=38, scale=1.0),
    "half_q23":  dict(crf=23, scale=0.5),
}

# Camera counts to sweep (clamped to the number of cameras actually available).
CAMERA_COUNTS = [1, 2, 4, 8, 16, 20]

# Total timesteps to pull per measurement (each timestep = one frame from EVERY
# camera in the subset = one synchronized multi-view bundle).
NUM_FRAMES = 30

# Bundles to treat as warm-up (excluded from T_steady). The 1st bundle is always
# T_first and is reported separately; WARMUP_BUNDLES additional early bundles are
# also excluded so T_steady reflects true steady-state per-bundle latency.
WARMUP_BUNDLES = 3

# Repeat the whole (open + decode) measurement this many times; report median.
NUM_REPEATS = 3

# --- Parallel CPU decode (DataLoader-style) --------------------------------
# Each view is decoded by a worker; every timestep is synchronized with a
# barrier (like a DataLoader collating one batch from num_workers before the
# next). Modes to sweep:
#   "sync"    -> single thread, sequential per camera (baseline, no workers)
#   "thread"  -> threading.Thread workers (PyAV releases the GIL while decoding)
#   "process" -> multiprocessing workers (true parallel; frame IPC overhead)
PARALLEL_MODES = ["sync", "thread", "process"]

# Worker-count sweep. Ints set a fixed pool size; "auto" means one worker per
# camera (== n_cams, the classic DataLoader "num_workers = #views" setup).
# When workers < cameras, each worker round-robins several views per timestep.
# Ignored for the "sync" mode (always 1).
WORKER_COUNTS = ["auto", 2, 4, 8]

# Only transfer frame shape across the process boundary (cheap), not pixels.
# True is the realistic streaming cost if pixels are needed in the main proc;
# False isolates pure decode throughput. Threads always see real frames.
PROCESS_RETURN_PIXELS = False

# --- Synthetic-data settings (only used when N3DV_SCENE_DIR is empty) -------
SYNTH_NUM_CAMERAS = 20      # N3DV has ~20 cameras
SYNTH_NUM_FRAMES = 90
SYNTH_RESOLUTION = (1352, 1014)  # N3DV native is 2704x2028; use /2 for speed
SYNTH_FPS = 30

import os
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("WORK_DIR :", WORK_DIR.resolve())
print("Scene dir:", N3DV_SCENE_DIR or "(none -> synthetic)")
print(f"NUM_FRAMES={NUM_FRAMES}  WARMUP_BUNDLES={WARMUP_BUNDLES}  NUM_REPEATS={NUM_REPEATS}")
print(f"PARALLEL_MODES={PARALLEL_MODES}  WORKER_COUNTS={WORKER_COUNTS}  cpu_count={os.cpu_count()}")
assert NUM_FRAMES > WARMUP_BUNDLES + 1, "Need >1 steady bundle after T_first + warm-up"

## Environment / decoder availability

In [ ]:
import shutil, subprocess, importlib, time, json
import numpy as np
import pandas as pd

FFMPEG = shutil.which("ffmpeg")
FFPROBE = shutil.which("ffprobe")
print("ffmpeg :", FFMPEG)
print("ffprobe:", FFPROBE)

try:
    import av
    print("PyAV   :", av.__version__, "(libav)")
except Exception as e:
    raise RuntimeError(
        "PyAV is required (this notebook uses the ffmpeg/libav decode path only). "
        "Install with: pip install av"
    ) from e

assert FFMPEG, "ffmpeg is required to build quality variants / synthetic data."

## Locate or synthesize source videos

Produces `SOURCE_VIDEOS`: an ordered list of per-camera mp4 paths.
If the scene only ships extracted PNGs (QUEEN's layout: `camNN/images/*.png`),
they are encoded once into a near-lossless mp4 so decoding can be timed uniformly.

In [ ]:
def run(cmd):
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def make_synthetic_videos():
    """Moving testsrc2 (motion + detail) per camera; cheap but realistic decode."""
    out_dir = WORK_DIR / "synthetic_source"
    out_dir.mkdir(parents=True, exist_ok=True)
    W, H = SYNTH_RESOLUTION
    paths = []
    for c in range(SYNTH_NUM_CAMERAS):
        p = out_dir / f"cam{c:02d}.mp4"
        paths.append(p)
        if p.exists():
            continue
        cmd = [
            FFMPEG, "-y", "-f", "lavfi",
            "-i", f"testsrc2=size={W}x{H}:rate={SYNTH_FPS}:duration={SYNTH_NUM_FRAMES/SYNTH_FPS:.4f}",
            "-vf", f"hue=h={c*17}",
            "-c:v", "libx264", "-preset", "medium", "-crf", "15",
            "-pix_fmt", "yuv420p", str(p),
        ]
        print("synth", p.name)
        run(cmd)
    return paths


def pngs_to_video(png_dir, out_path, fps=30):
    pngs = sorted(Path(png_dir).glob("*.png"))
    if not pngs:
        return None
    cmd = [
        FFMPEG, "-y", "-framerate", str(fps),
        "-i", str(Path(png_dir) / "%04d.png"),
        "-c:v", "libx264", "-preset", "medium", "-crf", "12",
        "-pix_fmt", "yuv420p", str(out_path),
    ]
    run(cmd)
    return out_path


def locate_source_videos():
    if not N3DV_SCENE_DIR:
        return make_synthetic_videos()
    scene = Path(N3DV_SCENE_DIR)
    assert scene.is_dir(), f"N3DV_SCENE_DIR not found: {scene}"

    # Case 1: original DyNeRF release -> cam00.mp4 .. camNN.mp4
    mp4s = sorted(scene.glob("cam*.mp4"))
    if mp4s:
        print(f"found {len(mp4s)} mp4 streams in {scene}")
        return mp4s

    # Case 2: QUEEN layout -> camNN/images/*.png ; encode once.
    cam_dirs = sorted(d for d in scene.glob("cam*") if d.is_dir())
    assert cam_dirs, f"No cam*.mp4 and no cam*/ dirs under {scene}"
    enc_dir = WORK_DIR / "encoded_from_png"
    enc_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for d in cam_dirs:
        png_dir = d / "images" if (d / "images").is_dir() else d
        out = enc_dir / f"{d.name}.mp4"
        if not out.exists():
            print("encode", d.name, "<-", png_dir)
            pngs_to_video(png_dir, out)
        if out.exists():
            paths.append(out)
    return paths


SOURCE_VIDEOS = locate_source_videos()
print(f"\n{len(SOURCE_VIDEOS)} source camera videos")
for p in SOURCE_VIDEOS[:3]:
    print("  ", p)

## Build quality variants

Re-encode every source stream into each `QUALITY_VARIANTS` setting once.
Resulting tree: `WORK_DIR/variants/<quality>/cam<NN>.mp4`.

In [ ]:
def file_mb(p):
    return Path(p).stat().st_size / 1e6


VARIANT_DIR = WORK_DIR / "variants"
variant_videos = {}  # quality -> [paths]
variant_stats = []

for q, params in QUALITY_VARIANTS.items():
    out_dir = VARIANT_DIR / q
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for src in SOURCE_VIDEOS:
        out = out_dir / Path(src).name
        if not out.exists():
            vf = []
            if params["scale"] != 1.0:
                vf.append(f"scale=iw*{params['scale']}:ih*{params['scale']}")
            cmd = [FFMPEG, "-y", "-i", str(src)]
            if vf:
                cmd += ["-vf", ",".join(vf)]
            cmd += [
                "-c:v", "libx264", "-preset", "medium",
                "-crf", str(params["crf"]), "-pix_fmt", "yuv420p", str(out),
            ]
            run(cmd)
        paths.append(out)
    variant_videos[q] = paths
    total_mb = sum(file_mb(p) for p in paths)
    variant_stats.append(dict(quality=q, crf=params["crf"], scale=params["scale"],
                              n_cams=len(paths), total_MB=round(total_mb, 1),
                              MB_per_cam=round(total_mb / len(paths), 2)))
    print(f"{q:>10}: {len(paths)} cams, {total_mb:7.1f} MB total")

pd.DataFrame(variant_stats)

## Decoders: sync baseline + parallel CPU workers (PyAV)

Implemented in [`mv_parallel_decode.py`](./mv_parallel_decode.py) (a real
module so `multiprocessing` workers are importable under fork **and** spawn).
`run_decode(paths, num_frames, warmup, mode, num_workers, ...)` dispatches on:

- **`mode="sync"`** — one thread, cameras decoded in order. The baseline; no
  workers (`n_workers=1`).
- **`mode="thread"`** — `threading` workers. PyAV releases the GIL during
  libav decode, so threads overlap; no IPC, but GIL-limited on heavy CPU.
- **`mode="process"`** — `multiprocessing` workers (fork). True parallel decode;
  pays frame IPC. `PROCESS_RETURN_PIXELS` toggles whether decoded pixels are
  shipped back to the parent (realistic streaming cost) or just the shape
  (isolates pure decode throughput).

**DataLoader-style synchronization.** Each worker owns one or more camera
streams (round-robin when `n_workers < n_cams`). Per timestep the main thread
releases all workers, then a **barrier** blocks the bundle until every worker
has produced frame `t`; only then does `t+1` start. Workers never run ahead, so
the measured per-bundle latency is the genuine synchronized multi-view cost —
exactly how a DataLoader collates one batch from `num_workers` before the next.

**`num_workers`**: an int sets a fixed pool; `"auto"` means one worker per
camera (`= n_cams`, the classic `num_workers = #views`). The same three phases
(`T_open` / `T_first` / `T_steady`) are measured for every mode.

In [ ]:
import importlib
import mv_parallel_decode as mvp
importlib.reload(mvp)  # pick up edits to the module without kernel restart

# Thin wrappers so the rest of the notebook stays decoder-agnostic.
run_decode = mvp.run_decode
resolve_workers = mvp.resolve_workers

# Backwards-compatible name: the original sync multi-view decoder.
def decode_multiview_bundles(paths, num_frames, warmup):
    return mvp.decode_sync(paths, num_frames, warmup)

# Quick self-check on the first 2 source videos (sanity, not timed).
_probe = SOURCE_VIDEOS[:2]
for _m in PARALLEL_MODES:
    r = run_decode(_probe, 6, 1, _m, num_workers="auto",
                    return_pixels=PROCESS_RETURN_PIXELS)
    print(f"{_m:>8}: bundles={r['bundles']} workers={r['n_workers']} "
          f"open={1000*r['t_open']:.1f}ms steady={1000*r['t_steady']:.2f}ms")

## Benchmark: quality × cameras × mode × workers, phase-split

Sweeps `quality × n_cams × parallel-mode × n_workers`. For each combination we
run `run_decode` `NUM_REPEATS` times and take the **median across repeats** of
each phase (`T_open`, `T_first`, `T_steady`).

Worker-count rules:
- `sync` always uses 1 worker (the `WORKER_COUNTS` axis is collapsed for it).
- `"auto"` resolves to `n_cams` (one worker per view).
- Fixed ints are clamped to `n_cams`; duplicate resolved counts are de-duped so
  e.g. `4` and `"auto"` at 4 cameras aren't measured twice.

`T_steady` is the streaming-critical number; `speedup_vs_sync` is added in the
tables section to show parallel efficiency.

In [ ]:
max_cams = len(SOURCE_VIDEOS)
cam_counts = sorted({min(c, max_cams) for c in CAMERA_COUNTS})
print("camera counts:", cam_counts, "(max available:", max_cams, ")")
print("modes:", PARALLEL_MODES, " worker sweep:", WORKER_COUNTS, "\n")

rows = []
for q in QUALITY_VARIANTS:
    vids = variant_videos[q]
    for n_cams in cam_counts:
        subset = vids[:n_cams]
        for mode in PARALLEL_MODES:
            if mode == "sync":
                worker_opts = [("sync", 1)]  # single worker, label = "sync"
            else:
                # Resolve + de-dupe worker counts for this camera count.
                seen, worker_opts = set(), []
                for wc in WORKER_COUNTS:
                    nw = resolve_workers(wc, n_cams)
                    if nw in seen:
                        continue
                    seen.add(nw)
                    label = "auto" if wc == "auto" else str(wc)
                    worker_opts.append((label, nw))

            for wlabel, nw in worker_opts:
                reps = [run_decode(subset, NUM_FRAMES, WARMUP_BUNDLES, mode,
                                    num_workers=nw,
                                    return_pixels=PROCESS_RETURN_PIXELS)
                        for _ in range(NUM_REPEATS)]
                t_open = float(np.median([r["t_open"] for r in reps]))
                t_first = float(np.median([r["t_first"] for r in reps]))
                t_steady = float(np.median([r["t_steady"] for r in reps]))
                steady_std = float(np.median([r["steady_std"] for r in reps]))
                bundles = int(np.median([r["bundles"] for r in reps]))
                # --- demux / decode split (same steady window as T_steady) ---
                t_demux_steady = float(np.median([r["t_demux_steady"] for r in reps]))
                t_decode_steady = float(np.median([r["t_decode_steady"] for r in reps]))
                demux_std = float(np.median([r["demux_steady_std"] for r in reps]))
                decode_std = float(np.median([r["decode_steady_std"] for r in reps]))
                t_demux_first = float(np.median([r["t_demux_first"] for r in reps]))
                t_decode_first = float(np.median([r["t_decode_first"] for r in reps]))

                rows.append(dict(
                    quality=q,
                    crf=QUALITY_VARIANTS[q]["crf"],
                    scale=QUALITY_VARIANTS[q]["scale"],
                    n_cams=n_cams,
                    mode=mode,
                    workers=wlabel,
                    n_workers=nw,
                    bundles=bundles,
                    T_open_s=round(t_open, 4),
                    T_first_s=round(t_first, 4),
                    T_steady_s=round(t_steady, 4),
                    T_steady_ms=round(1000 * t_steady, 2),
                    steady_std_ms=round(1000 * steady_std, 2),
                    per_cam_steady_ms=round(1000 * t_steady / n_cams, 3),
                    stream_fps=round(1.0 / t_steady, 2) if t_steady > 0 else float("inf"),
                    # --- new: pure demux vs pure decode (additive) ---
                    T_demux_steady_ms=round(1000 * t_demux_steady, 2),
                    T_decode_steady_ms=round(1000 * t_decode_steady, 2),
                    demux_std_ms=round(1000 * demux_std, 2),
                    decode_std_ms=round(1000 * decode_std, 2),
                    T_demux_first_ms=round(1000 * t_demux_first, 2),
                    T_decode_first_ms=round(1000 * t_decode_first, 2),
                    per_cam_decode_ms=round(1000 * t_decode_steady / n_cams, 3),
                    per_cam_demux_ms=round(1000 * t_demux_steady / n_cams, 3),
                ))
                print(f"{q:>9} | {n_cams:>2}cam | {mode:>7} | "
                      f"w={wlabel:>4}({nw:>2}) | "
                      f"open {1000*t_open:7.1f} | first {1000*t_first:7.1f} | "
                      f"steady {1000*t_steady:7.2f} ms "
                      f"(dmx {1000*t_demux_steady:6.2f} / dec {1000*t_decode_steady:6.2f}) "
                      f"(~{1.0/t_steady:6.1f} fps)")

df = pd.DataFrame(rows)
csv_path = WORK_DIR / "decode_timing_results.csv"
df.to_csv(csv_path, index=False)
print("\nsaved ->", csv_path, " rows:", len(df))
df

## Results: tables

In [ ]:
# Speedup vs the sync baseline (same quality + n_cams).
sync = (df[df["mode"] == "sync"]
        .set_index(["quality", "n_cams"])["T_steady_s"].rename("sync_steady_s"))
df = df.join(sync, on=["quality", "n_cams"])
df["speedup_vs_sync"] = (df["sync_steady_s"] / df["T_steady_s"]).round(2)

# A single "config" label for mode+workers.
df["config"] = df.apply(
    lambda r: "sync" if r["mode"] == "sync" else f"{r['mode']}/w={r['workers']}",
    axis=1)

# Pick a representative quality for the per-config breakdown tables.
QUAL = "high_q18" if "high_q18" in QUALITY_VARIANTS else list(QUALITY_VARIANTS)[0]
qd = df[df.quality == QUAL]
print(f"Tables below are for quality = {QUAL!r}\n")

print("=== T_steady (ms/bundle) — fused demux+decode, rows=config, cols=n_cams ===")
display(qd.pivot_table(index="config", columns="n_cams", values="T_steady_ms").round(2))

# --- pure decode vs pure demux split (additive; T_steady above unchanged) ---
# Note: for thread/process these are per-bundle MAX over workers, so
# demux + decode need not equal T_steady (bottleneck worker can differ).
# For sync they are the per-bundle SUM over cameras, so they add up to T_steady.
print("\n=== T_decode_steady (ms/bundle) — PURE decode (packet.decode) ===")
display(qd.pivot_table(index="config", columns="n_cams",
                       values="T_decode_steady_ms").round(2))

print("\n=== T_demux_steady (ms/bundle) — PURE demux (container.demux) ===")
display(qd.pivot_table(index="config", columns="n_cams",
                       values="T_demux_steady_ms").round(2))

print("\n=== speedup vs sync (×) — higher is better ===")
display(qd.pivot_table(index="config", columns="n_cams", values="speedup_vs_sync").round(2))

print("\n=== T_open (ms) — worker spin-up + decoder init ===")
display(qd.pivot_table(index="config", columns="n_cams", values="T_open_s")
          .mul(1000).round(1))

print("\n=== T_first (ms) — first synchronized bundle (cold) ===")
display(qd.pivot_table(index="config", columns="n_cams", values="T_first_s")
          .mul(1000).round(1))

print("\n=== sustainable streaming FPS (1 / T_steady) ===")
display(qd.pivot_table(index="config", columns="n_cams", values="stream_fps").round(1))

print("\n=== best config per (quality, n_cams) by T_steady ===")
best = (df.loc[df.groupby(["quality", "n_cams"])["T_steady_s"].idxmin()]
          [["quality", "n_cams", "config", "T_steady_ms",
            "T_decode_steady_ms", "T_demux_steady_ms",
            "speedup_vs_sync", "stream_fps"]]
          .sort_values(["quality", "n_cams"]))
display(best)

## Results: plots

In [ ]:
import matplotlib.pyplot as plt

QUAL = "high_q18" if "high_q18" in QUALITY_VARIANTS else list(QUALITY_VARIANTS)[0]
qd = df[df.quality == QUAL]
configs = sorted(qd["config"].unique(),
                 key=lambda c: (c != "sync", c))  # sync first

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# (1) T_steady vs camera count, one line per config
ax = axes[0]
for cfg in configs:
    s = qd[qd.config == cfg].sort_values("n_cams")
    ax.errorbar(s.n_cams, s.T_steady_ms, yerr=s.steady_std_ms,
                marker="o", capsize=3, label=cfg)
ax.set_xlabel("# cameras"); ax.set_ylabel("T_steady (ms / bundle)")
ax.set_title(f"Warm bundle latency  [{QUAL}]"); ax.legend(fontsize=8); ax.grid(alpha=.3)

# (2) speedup vs sync at the largest camera count
ax = axes[1]
big = qd[qd.n_cams == qd.n_cams.max()].sort_values("speedup_vs_sync")
ax.barh(big.config, big.speedup_vs_sync)
ax.axvline(1.0, color="k", ls="--", lw=1, label="sync baseline")
ax.set_xlabel("speedup vs sync (×)")
ax.set_title(f"Parallel efficiency @ {qd.n_cams.max()} cams  [{QUAL}]")
ax.legend(fontsize=8); ax.grid(alpha=.3, axis="x")

# (3) T_open vs camera count (worker spin-up cost) per config
ax = axes[2]
for cfg in configs:
    s = qd[qd.config == cfg].sort_values("n_cams")
    ax.plot(s.n_cams, s.T_open_s * 1000, marker="s", label=cfg)
ax.set_xlabel("# cameras"); ax.set_ylabel("T_open (ms)")
ax.set_title(f"Open + worker spin-up  [{QUAL}]")
ax.legend(fontsize=8); ax.grid(alpha=.3)

plt.tight_layout()
plt.savefig(WORK_DIR / "decode_timing.png", dpi=120, bbox_inches="tight")
plt.show()

# Bonus: quality × best-mode steady latency at full camera rig.
fig2, ax = plt.subplots(figsize=(8, 4.5))
full = df[df.n_cams == df.n_cams.max()]
for cfg in sorted(full["config"].unique(), key=lambda c: (c != "sync", c)):
    s = full[full.config == cfg].set_index("quality").reindex(
        list(QUALITY_VARIANTS)).reset_index()
    ax.plot(s.quality, s.T_steady_ms, marker="o", label=cfg)
ax.set_ylabel("T_steady (ms / bundle)"); ax.set_xlabel("quality")
ax.set_title(f"Quality vs warm latency @ {df.n_cams.max()} cams")
ax.tick_params(axis="x", rotation=30); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(WORK_DIR / "decode_timing_quality.png", dpi=120, bbox_inches="tight")
plt.show()

## Streaming-3DGS interpretation

`T_steady` is the per-timestep cost the online 3DGS loop pays once warm. We
compare it against the training-FPS budget to flag where decoding becomes the
bottleneck, and report `T_open` / `T_first` as the one-time startup latency the
pipeline absorbs before steady streaming begins.

In [ ]:
TARGET_TRAIN_FPS = 1.0  # timesteps/sec the 3DGS fitter can sustain
budget_s = 1.0 / TARGET_TRAIN_FPS

print(f"Decode budget per timestep @ {TARGET_TRAIN_FPS} train-fps = {budget_s:.3f}s\n")

# For each (quality, n_cams): does the BEST config fit the budget, and which is it?
g = df.loc[df.groupby(["quality", "n_cams"])["T_steady_s"].idxmin()].copy()
g["fits_budget"] = np.where(g["T_steady_s"] <= budget_s, "ok", "<<BOTTLENECK")
print("=== best config per (quality, n_cams) vs budget ===")
display(g[["quality", "n_cams", "config", "T_steady_ms",
           "speedup_vs_sync", "stream_fps", "fits_budget"]]
        .sort_values(["quality", "n_cams"]).reset_index(drop=True))

# Parallel scaling summary at the full camera rig.
full = df[df.n_cams == df.n_cams.max()]
fast = df.loc[df.T_steady_s.idxmin()]
slow = df.loc[df.T_steady_s.idxmax()]
print(f"\nFastest overall : {fast['quality']} | {fast['config']} | "
      f"{fast['n_cams']} cams -> {fast['T_steady_ms']} ms/bundle "
      f"({fast['stream_fps']} fps, {fast['speedup_vs_sync']}x vs sync)")
print(f"Slowest overall : {slow['quality']} | {slow['config']} | "
      f"{slow['n_cams']} cams -> {slow['T_steady_ms']} ms/bundle "
      f"({slow['stream_fps']} fps)")

# Mean parallel speedup per mode (across all quality / n_cams / workers).
print("\n=== mean speedup vs sync, by mode ===")
display(df[df["mode"] != "sync"]
        .groupby("mode")["speedup_vs_sync"]
        .agg(["mean", "max"]).round(2))

# One-time startup latency (T_open + T_first) of the best config @ max cams.
print("\nStartup latency (T_open + T_first) of best config @ max cameras:")
bf = full.loc[full.groupby("quality")["T_steady_s"].idxmin()]
for _, r in bf.sort_values("quality").iterrows():
    print(f"  {r['quality']:>10} [{r['config']:>14}]: "
          f"{1000*r['T_open_s']:7.1f} ms open + {1000*r['T_first_s']:7.1f} ms first "
          f"= {1000*(r['T_open_s']+r['T_first_s']):7.1f} ms before steady streaming")